# 03 - Swin Transformer

**Goal:** Understand Swin Transformer - the "Swin" in your production SwinUNETR.

---

## Why Swin?

ViT problems:
1. Global attention is expensive: O(n²) where n = number of patches
2. No hierarchy: all patches same resolution
3. Fixed input size

**Swin Transformer** (Shifted WINdows) solves these:
1. **Windowed attention**: attend only within local windows → O(n)
2. **Hierarchical features**: like a CNN, merge patches at each stage
3. **Shifted windows**: allows cross-window information flow

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

## Key Idea 1: Window Attention

Instead of global attention (all patches attend to all), divide into windows:

```
Image: 8x8 patches
Windows: 4 windows of 4x4 patches each

┌───────┬───────┐
│ Win 1 │ Win 2 │
│ 4x4   │ 4x4   │
├───────┼───────┤
│ Win 3 │ Win 4 │
│ 4x4   │ 4x4   │
└───────┴───────┘

Each window does self-attention independently.
```

Complexity: O(n) instead of O(n²)!

In [ ]:
# Complexity comparison
def global_attention_complexity(num_patches):
    return num_patches ** 2

def window_attention_complexity(num_patches, window_size):
    num_windows = num_patches // (window_size ** 2)
    per_window = (window_size ** 2) ** 2
    return num_windows * per_window

# 224x224 image with 4x4 patches = 56x56 = 3136 patches
num_patches = 56 * 56
window_size = 7  # Typical Swin window size

global_ops = global_attention_complexity(num_patches)
window_ops = window_attention_complexity(num_patches, window_size)

print(f"Patches: {num_patches}")
print(f"Global attention: {global_ops:,} operations")
print(f"Window attention (7x7): {window_ops:,} operations")
print(f"Reduction: {global_ops / window_ops:.0f}x faster!")

In [ ]:
# Visualize windows
def visualize_windows(size=8, window_size=4):
    """Show how image is divided into windows."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Regular windows
    img = np.zeros((size, size))
    colors = plt.cm.Set3(np.linspace(0, 1, (size // window_size) ** 2))
    
    window_idx = 0
    for i in range(0, size, window_size):
        for j in range(0, size, window_size):
            img[i:i+window_size, j:j+window_size] = window_idx
            window_idx += 1
    
    axes[0].imshow(img, cmap='Set3')
    axes[0].set_title(f'Regular Windows\n({window_size}x{window_size} each)')
    for i in range(0, size + 1, window_size):
        axes[0].axhline(y=i-0.5, color='black', linewidth=2)
        axes[0].axvline(x=i-0.5, color='black', linewidth=2)
    axes[0].set_xticks(range(size))
    axes[0].set_yticks(range(size))
    
    # Shifted windows
    shift = window_size // 2
    img_shifted = np.zeros((size, size))
    
    # Shift and wrap
    window_idx = 0
    for i in range(-shift, size - shift, window_size):
        for j in range(-shift, size - shift, window_size):
            i_start = i % size
            j_start = j % size
            i_end = min(i + window_size, size) % size or size
            j_end = min(j + window_size, size) % size or size
            
            # This is simplified - real implementation handles wrapping
            for ii in range(max(0, i), min(size, i + window_size)):
                for jj in range(max(0, j), min(size, j + window_size)):
                    img_shifted[ii, jj] = window_idx
            window_idx += 1
    
    axes[1].imshow(img_shifted, cmap='Set3')
    axes[1].set_title(f'Shifted Windows\n(shifted by {shift})')
    for i in range(shift, size + 1, window_size):
        axes[1].axhline(y=i-0.5, color='black', linewidth=2)
    for j in range(shift, size + 1, window_size):
        axes[1].axvline(x=j-0.5, color='black', linewidth=2)
    # Add boundary lines
    axes[1].axhline(y=-0.5, color='black', linewidth=2)
    axes[1].axvline(x=-0.5, color='black', linewidth=2)
    axes[1].set_xticks(range(size))
    axes[1].set_yticks(range(size))
    
    plt.tight_layout()
    plt.show()

visualize_windows(size=8, window_size=4)

## Key Idea 2: Shifted Windows

Problem with window attention: no information flows between windows!

**Solution:** Alternate between regular and shifted windows:

```
Layer 1: Regular windows
┌───┬───┐
│ A │ B │
├───┼───┤
│ C │ D │
└───┴───┘

Layer 2: Shifted windows (shift by half window size)
    ┌───┬───┐
  ┌─┼───┼───┼─┐
  │ │   │   │ │  ← Windows now cross the previous boundaries
  ├─┼───┼───┼─┤
  │ │   │   │ │
  └─┼───┼───┼─┘
    └───┴───┘
```

This allows information to flow across the entire image over multiple layers.

## Key Idea 3: Hierarchical Features

Like CNNs, Swin creates multi-scale features by merging patches:

```
Stage 1: 56x56 patches, 96 channels
    ↓ Patch Merging (2x2 → 1)
Stage 2: 28x28 patches, 192 channels
    ↓ Patch Merging
Stage 3: 14x14 patches, 384 channels
    ↓ Patch Merging
Stage 4: 7x7 patches, 768 channels
```

This is crucial for U-Net - we need features at different scales for the decoder!

In [ ]:
class PatchMerging(nn.Module):
    """Merge 2x2 neighboring patches → reduce spatial, increase channels."""
    
    def __init__(self, dim):
        super().__init__()
        self.reduction = nn.Linear(4 * dim, 2 * dim, bias=False)
        self.norm = nn.LayerNorm(4 * dim)
    
    def forward(self, x):
        # x: (batch, H*W, C)
        batch, L, C = x.shape
        H = W = int(L ** 0.5)
        
        x = x.view(batch, H, W, C)
        
        # Take 2x2 patches
        x0 = x[:, 0::2, 0::2, :]  # Top-left
        x1 = x[:, 1::2, 0::2, :]  # Bottom-left
        x2 = x[:, 0::2, 1::2, :]  # Top-right
        x3 = x[:, 1::2, 1::2, :]  # Bottom-right
        
        x = torch.cat([x0, x1, x2, x3], dim=-1)  # (batch, H/2, W/2, 4*C)
        x = x.view(batch, -1, 4 * C)
        
        x = self.norm(x)
        x = self.reduction(x)  # 4*C → 2*C
        
        return x

# Test
merge = PatchMerging(dim=96)
x = torch.randn(1, 56 * 56, 96)  # 56x56 patches, 96 channels

output = merge(x)
print(f"Input: {x.shape}")
print(f"Output: {output.shape}")  # Should be (1, 28*28, 192)

## Swin Architecture Overview

```
Input Image (H x W x 3)
    ↓
Patch Partition (4x4 patches) → H/4 x W/4 x 48
    ↓
Linear Embedding → H/4 x W/4 x C
    ↓
Stage 1: Swin Blocks (window attention + shifted window attention)
    ↓
Patch Merging → H/8 x W/8 x 2C
    ↓
Stage 2: Swin Blocks
    ↓
Patch Merging → H/16 x W/16 x 4C
    ↓
Stage 3: Swin Blocks
    ↓
Patch Merging → H/32 x W/32 x 8C
    ↓
Stage 4: Swin Blocks
```

In [ ]:
# Feature map sizes through Swin (for 224x224 input)
print("Swin Transformer feature sizes (224x224 input):")
print()
stages = [
    ("After patch partition", "56x56", "C=96"),
    ("After Stage 1", "56x56", "C=96"),
    ("After Patch Merge 1", "28x28", "C=192"),
    ("After Stage 2", "28x28", "C=192"),
    ("After Patch Merge 2", "14x14", "C=384"),
    ("After Stage 3", "14x14", "C=384"),
    ("After Patch Merge 3", "7x7", "C=768"),
    ("After Stage 4", "7x7", "C=768"),
]

for name, spatial, channels in stages:
    print(f"{name:25s} → {spatial:8s} × {channels}")

## Swin for 3D (Medical Imaging)

Your production code uses **3D Swin Transformer**:

- Window size: 7x7x7 (instead of 7x7)
- Patches: 4x4x4 or 2x2x2
- Same shifted window concept, but in 3D

From your config:
```yaml
model:
  input_shape: [96, 96, 96]  # 3D volume
  features_chan: 24         # Swin feature size
```

## Why Swin Works Well for Medical Imaging

1. **Efficient**: Window attention handles large 3D volumes
2. **Hierarchical**: Multi-scale features for different organ sizes
3. **Long-range**: Shifted windows capture global context
4. **Flexible**: Works with variable input sizes

## Summary

| Concept | What it means |
|---------|---------------|
| **Window attention** | Attend only within local windows → efficient |
| **Shifted windows** | Alternate window positions → cross-window info flow |
| **Patch merging** | Combine 2x2 patches → hierarchical features |
| **Stages** | Multiple resolution levels (like CNN) |

**Swin = Efficient hierarchical transformer for vision**

**Next:** SwinUNETR - combining Swin encoder with U-Net decoder.